# QVerse — Introduction to Quantum Computing & Programming
        ## Week 9: Quantum Protocols — Teleportation and Superdense Coding

        **Level:** Beginner  
        **Recommended study time:** 2–4 hours  
        **Prerequisites:** Weeks 1–8

        ### Learning objectives
        - Explain what quantum teleportation transfers.
- Implement a coherent/deferred-measurement teleportation circuit.
- Implement superdense coding.
- Identify the role of entanglement and classical information.

        ---
        **How to use this notebook**

        1. Read the short theory sections.
        2. Make a prediction before running each guided experiment.
        3. Run and modify the code.
        4. Complete every **TODO** exercise.
        5. Finish the reflection section in your own words.

        The goal is not to memorize syntax. The goal is to connect **quantum idea → circuit → result → explanation**.

In [ ]:
# Run this only if your environment does not have the required packages.
# In a terminal, the preferred setup is:
# python -m pip install "qiskit[visualization]>=2.5" matplotlib numpy

# In a fresh Colab notebook you can instead uncomment:
# %pip install "qiskit[visualization]>=2.5" matplotlib numpy -q

## 1. Teleportation in one sentence

Quantum teleportation transfers an **unknown quantum state** from Alice to Bob using:
1. a pre-shared Bell pair;
2. local quantum operations;
3. two classical bits of communication.

It does not transport matter and cannot transmit information faster than light.

In this notebook we first use a **deferred-measurement/coherent correction** version. This avoids mid-circuit classical control while preserving the logic of the protocol.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity

def teleportation_circuit(theta):
    qc = QuantumCircuit(3)

    # q0: message
    qc.ry(theta, 0)

    # q1-q2: Bell pair shared by Alice and Bob
    qc.h(1)
    qc.cx(1, 2)

    # Bell-basis operations on Alice's qubits
    qc.cx(0, 1)
    qc.h(0)

    # Coherent versions of the classical corrections
    qc.cx(1, 2)
    qc.cz(0, 2)

    return qc

## 2. Verify that Bob receives the message state

In [ ]:
for theta in [0.0, 0.3, 1.0, np.pi/2, np.pi]:
    target = QuantumCircuit(1)
    target.ry(theta, 0)
    target_state = Statevector.from_instruction(target)

    full = Statevector.from_instruction(teleportation_circuit(theta))
    bob_state = partial_trace(full, [0, 1])

    print(theta, state_fidelity(bob_state, target_state))

## 3. Inspect the circuit

In [ ]:
teleportation_circuit(0.8).draw("mpl")

In the textbook circuit, Alice measures two qubits and sends two classical bits to Bob. Bob applies X and Z corrections based on those bits. The coherent form above replaces that classical feed-forward with controlled quantum gates, which is convenient for ideal local simulation.

## 4. Superdense coding

In [ ]:
from qiskit.primitives import StatevectorSampler

def superdense_coding(message):
    if message not in {"00", "01", "10", "11"}:
        raise ValueError("message must be two bits")

    qc = QuantumCircuit(2)

    # Shared Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Alice encodes two classical bits on q0.
    # Convention: first bit -> Z, second bit -> X
    if message[0] == "1":
        qc.z(0)
    if message[1] == "1":
        qc.x(0)

    # Bob decodes
    qc.cx(0, 1)
    qc.h(0)
    qc.measure_all()
    return qc

for message in ["00", "01", "10", "11"]:
    qc = superdense_coding(message)
    counts = StatevectorSampler(seed=1).run([qc], shots=100).result()[0].data.meas.get_counts()
    print(message, counts)

**Ordering note:** compare the requested message with Qiskit's displayed classical bitstring and explain the convention you observe. Do not silently assume left-to-right qubit order.

## Core exercises
1. Draw the teleportation circuit and label message/Alice/Bob qubits.
2. Test teleportation for at least five values of `theta` and report fidelity.
3. Explain the role of the two classical bits in textbook teleportation.
4. Implement superdense coding for all four messages and explain the observed bit ordering.

In [ ]:
# TODO: Write your solutions here.
# Add extra code cells when useful.

## Optional stretch challenge
Generate random one-qubit pure states using random `Ry` and `Rz` angles. Teleport at least 20 states and report the minimum fidelity in ideal simulation.

In [ ]:
# OPTIONAL TODO: Attempt the stretch challenge here.

## Weekly reflection
- What exactly is teleported?
- Why is pre-shared entanglement not enough by itself?
- Why does teleportation not enable faster-than-light communication?

## Submission checklist
- [ ] I made at least one prediction before executing a circuit.
- [ ] All guided examples run.
- [ ] I completed the core exercises.
- [ ] I explained the important output rather than only displaying it.
- [ ] My notebook is readable from top to bottom.